In [4]:
# @title FPDS Data Scraper
# @markdown ### 1. Configure Search
# @markdown Enter your search terms just like you would on FPDS.gov (e.g., `software`, `department_name:"DEPT OF DEFENSE"`).
QUERY = "software" # @param {type:"string"}

# @markdown ### 2. Date Range (Optional)
# @markdown Leave empty to ignore dates. Format: `YYYY-MM-DD`
START_DATE = "" # @param {type:"string"}
END_DATE = "" # @param {type:"string"}

## 3. Run Settings
High `BATCH_SIZE` (e.g. 5000-30000) is faster but may timeout on slow connections.
BATCH_SIZE = 5000
MAX_RECORDS = 10000
SLEEP_SECONDS = 2.0
OUTPUT_BASENAME = "FPDS_Awards"

import requests
import pandas as pd
import time
import json
import os
import re
from datetime import datetime
from urllib.parse import quote
from google.colab import files

def get_fpds_data():
    """
    Main execution function to scrape FPDS, process data, and generate output files.
    """
    print(f"Starting FPDS Scrape for query: '{QUERY}'")

    # --- 1. Setup & Configuration ---
    # Define the base FPDS export endpoint, build the final query string (including optional
    # date filters), and prepare headers to consistently receive CSV output.
    BASE_URL = "https://www.fpds.gov/ezsearch/fpdsportal"

    # Construct the query string handling optional dates
    # FPDS supports 'SIGNED_DATE:[YYYY/MM/DD, YYYY/MM/DD]' syntax in the query
    full_query = QUERY
    if START_DATE and END_DATE:
        # Convert YYYY-MM-DD to YYYY/MM/DD for FPDS syntax
        s_fpds = START_DATE.replace("-", "/")
        e_fpds = END_DATE.replace("-", "/")
        full_query += f' SIGNED_DATE:[{s_fpds}, {e_fpds}]'
    elif START_DATE or END_DATE:
        print("Warning: Both Start and End date must be provided to use date filtering. Ignoring dates.")

    encoded_query = quote(full_query)

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Accept': 'text/csv,application/json,application/xml'
    }

    # --- 2. Retrieval Loop ---
    # Page through the export results, respect MAX_RECORDS limits, and collect raw rows while
    # deduplicating on a stable transaction identifier.
    extracted_records = []
    seen_ids = set()
    start_index = 0
    total_fetched = 0

    # Session for connection reuse
    session = requests.Session()
    session.headers.update(headers)

    while True:
        # Check user limits
        if MAX_RECORDS and total_fetched >= MAX_RECORDS:
            print(f"Reached user-defined limit of {MAX_RECORDS} records.")
            break

        # Calculate fetch size (don't over-fetch if close to limit)
        current_fetch_size = BATCH_SIZE
        if MAX_RECORDS and (MAX_RECORDS - total_fetched) < BATCH_SIZE:
            current_fetch_size = MAX_RECORDS - total_fetched

        # Construct URL
        # We use templateName=CSV to get the file export directly
        # indexName=awardfull targets the full contract dataset
        url = (f"{BASE_URL}?s=FPDS.GOV&indexName=awardfull&templateName=CSV"
               f"&q={encoded_query}&start={start_index}&length={current_fetch_size}")

        print(f"? Fetching records {start_index} to {start_index + current_fetch_size}...", end=" ")

        try:
            response = session.get(url, timeout=45)

            if response.status_code != 200:
                print()
                print(f"? Error: HTTP {response.status_code}. Stopping.")
                break

            # If the response is HTML (login page or error), we assume failure
            if b"<!DOCTYPE html>" in response.content[:100]:
                print()
                print("? Error: Server returned HTML instead of CSV. You may be rate-limited or the query is invalid.")
                break

            # Parse CSV content
            # Skip malformed lines so one bad row does not stop the run
            from io import StringIO
            csv_data = StringIO(response.text)

            # Read CSV into Pandas
            # FPDS CSVs can have variable headers; we assume standard layout
            batch_df = pd.read_csv(csv_data, on_bad_lines='skip', low_memory=False)

            if batch_df.empty:
                print("? No more records found.")
                break

            records_in_batch = 0

            # --- 3. Deduplication & Processing ---
            # Normalize keys, select a stable ID, attach metadata for traceability,
            # and store the raw row for downstream transformation.
            for _, row in batch_df.iterrows():
                # Locate a stable ID. 'unique_transaction_id' is standard, fallback to 'contract_id' then 'piid'
                # Note: CSV columns are often capitalized or contain spaces
                record = row.to_dict()

                # Normalize keys slightly for easier access
                record_norm = {k.lower().replace(' ', '_'): v for k, v in record.items()}

                uid = record_norm.get('unique_transaction_id') or record_norm.get('contract_id') or record_norm.get('piid')

                if uid and uid in seen_ids:
                    continue

                if uid:
                    seen_ids.add(uid)

                # Add metadata
                record['_meta_scraped_at'] = datetime.utcnow().isoformat()
                record['_meta_source_url'] = url

                extracted_records.append(record)
                records_in_batch += 1
                total_fetched += 1

            print(f"? Got {records_in_batch} new unique records. (Total: {total_fetched})")

            # Stop if we got fewer records than requested (end of data)
            if len(batch_df) < current_fetch_size:
                print("? End of result set reached.")
                break

            start_index += len(batch_df)
            time.sleep(SLEEP_SECONDS)

        except Exception as e:
            print()
            print(f"? Critical Error: {e}")
            time.sleep(5) # Cooldown
            break

    # --- 4. Flattening & Export ---
    # Map raw FPDS fields into a stable schema, parse money fields, and output to Excel + JSONL.
    if not extracted_records:
        print("No records were retrieved. Check your query.")
        return

    print()
    print("Processing and flattening data for Excel...")

    # Normalize columns for the 'Clean' content map
    # We map flexible CSV headers to the user's requested schema
    # Use a helper to safely get columns ignoring case
    def get_val(row, legacy_keys):
        for k in legacy_keys:
            # Try exact match
            if k in row:
                return row[k]
            # Try normalized match
            n_k = k.lower().replace(' ','_')
            for rk in row.keys():
                if rk.lower().replace(' ','_') == n_k:
                    return row[rk]
        return None

    def parse_money(value):
        """
        Convert a raw money field into (amount, currency).
        Currency is inferred from symbols/codes when present; defaults to USD when amount exists.
        """
        if value is None or (isinstance(value, float) and pd.isna(value)):
            return None, None
        raw = str(value).strip()
        if not raw:
            return None, None

        currency = None
        currency_markers = {
            'USD': 'USD',
            '$': 'USD',
            'EUR': 'EUR',
            'GBP': 'GBP',
            'CAD': 'CAD',
            'AUD': 'AUD',
            'JPY': 'JPY'
        }
        for marker, code in currency_markers.items():
            if marker in raw:
                currency = code
                break

        cleaned = re.sub(r'[A-Za-z$(),]', '', raw)
        cleaned = cleaned.replace(',', '').strip()
        try:
            amount = float(cleaned) if cleaned else None
        except ValueError:
            amount = None

        if amount is not None and currency is None:
            currency = 'USD'

        return amount, currency

    flattened_rows = []

    for r in extracted_records:
        # Helper to grab fields robustly
        def grab(*keys):
            return get_val(r, keys)

        # Money parsing (obligated_amount_raw -> amount + currency)
        # Matches 'Action Obligation ($)' or 'obligated_amount'
        obs_amt = grab('action_obligation_($)', 'obligated_amount', 'dollars_obligated')
        amount_value, currency_value = parse_money(obs_amt)
        if amount_value is None:
            amount_value = 0.0

        item = {
            'award_id': grab('contract_id', 'unique_transaction_id', 'piid'),
            'source_site': 'FPDS',
            'contracting_agency_name': grab('contracting_agency', 'agency_name'),
            'awarding_sub_agency_name': grab('contracting_office_name', 'contracting_office'),
            'recipient_name': grab('legal_business_name', 'vendor_name', 'awardee'),
            'recipient_uei_duns': grab('unique_entity_id', 'uei', 'duns_number'),
            'naics_code': grab('naics', 'naics_code'),
            'psc_code': grab('psc', 'product_or_service_code'),
            'piid': grab('contract_id', 'piid'),
            'referenced_idv_piid': grab('reference_idv', 'referenced_idv_piid'),
            'action_date': grab('action_date'),
            'signed_date': grab('date_signed', 'signed_date'),
            'period_of_performance_start': grab('effective_date'),
            'period_of_performance_end': grab('completion_date', 'current_completion_date'),
            'obligated_amount_raw': obs_amt,
            'amount': amount_value,
            'currency': currency_value,
            'contract_type': grab('award/idv_type', 'type_of_contract', 'contract_pricing_type'),
            'place_of_performance_state': grab('place_of_performance_state'),
            'place_of_performance_country': grab('place_of_performance_country'),
            'description': grab('psc_description', 'description_of_requirement', 'naics_description'),
            'source_url': r.get('_meta_source_url'),
            'scraped_at_utc': r.get('_meta_scraped_at'),
            # Include the raw JSON string as requested
            'raw_json': json.dumps(r, default=str)
        }
        flattened_rows.append(item)

    # create DataFrames
    final_df = pd.DataFrame(flattened_rows)

    # Drop columns that are completely empty (all NaN/None)
    # This fulfills the request to "remove if that is not the data we can get"
    final_df.dropna(axis=1, how='all', inplace=True)

    # Save files
    csv_name = f"{OUTPUT_BASENAME}.csv"

    print(f"Saving {csv_name} (CSV)...")
    final_df.to_csv(csv_name, index=False)

    print()
    print("Done! Triggering download...")
    files.download(csv_name)

# Run the scraper
get_fpds_data()


SyntaxError: invalid syntax (1096491274.py, line 12)